# Smoothing Design Sweep (v5) — PandaOmron + GRPO LoRA

**What v4 could not answer.** v4 measured two isolated checkpoints (`iter_0011`,
`iter_0017`) and proved that roughening happens: endpoint `HF(â)` at τ=0 went
`0.0016 → 0.3116`, a 194× increase, with scale-controlled path jerk up 7.6×. That
is enough to justify building a constraint. It is **not** enough to size one,
because both knobs depend on the *shape* of `HF(iteration)`, which nobody has
measured:

| knob | what it needs | why two checkpoints can't give it |
|---|---|---|
| `smooth_coef` | the excess at the iteration where roughening **starts** | a hinge is linear in excess; sizing it from the endpoint value makes it reach 1% authority only at 4× threshold and 5% at 17× — possibly long after the damage |
| `smooth_hf_ref_scale` | where HF sits while success is still **improving** | 4.0 was a guess; if HF crosses 4× base at iteration 3 the threshold is too tight, if at iteration 15 it is too loose |
| instrument choice | which metric **leads** the success decline | correlation needs a series, and a constraint that fires simultaneously with the collapse cannot prevent it |

**What this notebook produces.**

1. `HF(â)` at τ=0 across a whole training trajectory (`iter_0001 … iter_0016` of
   the unconstrained run), overlaid with the same measurement on the
   `smooth_coef=0.15` run — a direct read on whether the term bent the curve.
2. Alignment of every roughness instrument against per-iteration success rate,
   with **lead time**: does roughness rise before success falls?
3. A dose–response grid over `(smooth_coef, smooth_hf_ref_scale)` using the run's
   own `|clip_loss|`, so the coefficient is read off measured excess instead of
   extrapolated from one endpoint.
4. A trainer-parity check: what this notebook computes at τ=0 versus what
   `smooth/hf_mean` logged live.
5. Cross-observation spread of the *base* threshold — the risk in a single global
   scalar `hf_ref`.

**Cost.** `HF_N_SEEDS × (4 denoise + 6 τ-probe)` forwards per checkpoint = 250 at
the defaults. 21 checkpoints + base ≈ 5.5k forwards, roughly 1–1.5 h on one GPU.
Results are cached to JSON after every checkpoint, so the sweep is resumable and
every analysis cell re-runs with no GPU.

**Nothing here runs the simulator.** No success rates are produced; the success
numbers come from the runs' TensorBoard logs.

In [ ]:
# Cell 1: Imports + config
import sys, os, json, time, glob, math
import numpy as np
import torch
import matplotlib.pyplot as plt

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from scripts.denoising_lab.denoising_lab import DenoisingLab, TrajectoryVisualizer

MODEL_PATH = "nvidia/GR00T-N1.6-3B"
EMBODIMENT_TAG = "robocasa_panda_omron"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EEF_KEY = "end_effector_position"

# LoRA config — MUST match what the checkpoints were trained with (GRPOConfig
# defaults). A mismatch here loads shapes that silently do not correspond.
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0

# ----------------------------------------------------------------- observations
# The trajectory sweep runs on ONE observation, which is what makes 21 checkpoints
# affordable. step010 is the init state both GRPO runs are pinned to via
# --init-state-npz-path, so it is the observation whose roughness the training
# telemetry actually describes.
OBS_MAIN = "/home/ubuntu/results/npz_save_CoffeeServeMug_v2/ep000_step010.npz"

# Extra observations for the global-scalar-threshold safety check at the end. v4
# found base HF(a_hat) at tau=0 differs 4.2x between step010 (0.0016) and step012
# (0.0067) — that spread is the entire risk in a single scalar hf_ref, and it is
# why report() in v4 printed "a global scalar is NOT safe".
OBS_EXTRA = [
    "/home/ubuntu/results/npz_save_CoffeeServeMug_v2/ep000_step012.npz",
]

# --------------------------------------------------------------------- series
_CTL = ("grpo_data/overfit_step10_v5_lr5.9e-5_PAWS_plam0.25_nlam0.05_loclip0.08"
        "_hiclip0.2_kll0.1_klb0.1_tratio1.75_mbrenorm_BMPARD_nojitterpair"
        "_1epoch_gs12_ng4_mb64")
_SMOOTH = ("grpo_data/overfit_step10_v5_lr5.9e-5_PAWS_plam0.25_nlam0.05_loclip0.08"
           "_hiclip0.2_kll0.1_klb0.1_tratio1.75_mbrenorm_BMPARD_nojitterpair"
           "_1epoch_gs12_ng4_IAG0.15_smooth0.15_mb64")

# name -> (run dir relative to REPO_ROOT, iterations wanted or None = all on disk)
#
# "unconstrained" is the trajectory that SIZES the constraint: same config, no
# smoothness term, 16 iterations spanning a success peak at iter 12 (0.75) and a
# decline to 0.54 by iter 16. "smooth0.15" is the same config WITH the term;
# overlaying them on one axis is the only direct read available on whether the
# term changed the field at all. Iterations are auto-discovered from disk, so a
# longer smooth0.15 run is picked up without editing this.
SERIES = {
    "unconstrained": (_CTL, None),
    "smooth0.15": (_SMOOTH, None),
}

HF_N_SEEDS = 25   # 25 x (4 denoise + len(TAU_CENTERS) probes) forwards per ckpt

# Cache. Written after EVERY checkpoint: a 1h sweep that dies at checkpoint 19
# should not cost the first 18. Re-running the sweep cell skips what is present.
RESULTS_JSON = os.path.join(os.getcwd(), "smoothing_sweep_results.json")

print(f"Device: {DEVICE}")
print(f"repo root: {REPO_ROOT}")
print(f"results cache: {RESULTS_JSON}")
for _name, (_dir, _want) in SERIES.items():
    _p = os.path.join(REPO_ROOT, _dir)
    print(f"  {_name:14s} {'OK ' if os.path.isdir(_p) else 'MISSING'} {_dir[-58:]}")

In [ ]:
# Cell 2: Training telemetry — read the runs' own TensorBoard scalars.
#
# The roughness measured below is a property of the WEIGHTS; success rate,
# |clip_loss| and ref_mse are properties of the RUN. Sizing a coefficient needs
# both on the same iteration axis: the coefficient converts an HF excess into a
# fraction of the policy loss, so |clip_loss| is the denominator, and the success
# curve is what tells us which iteration the constraint must act before.
#
# Falls back to literals (same numbers, extracted from these logs) if tensorboard
# does not import in this kernel, so every analysis cell stays runnable.
TB_TAGS = ["episode/success_rate", "train/clip_loss", "ref_mse/mean",
           "ref_mse/log_base_ratio_mean", "train/clipfrac", "smooth/hf_mean",
           "smooth/active_frac", "smooth/excess_mean", "smooth/loss",
           "smooth/hf_max", "lora/weight_delta_norm"]


def read_tb(run_dir):
    """{tag: {iteration: value}} merged across every event file in tb_logs/.

    Merged rather than taking the newest file because a resumed run writes a
    SECOND events file covering later iterations only (the unconstrained run
    splits 1-7 / 8-16 across two files). Reading one file silently truncates the
    trajectory to half of it.
    """
    from tensorboard.backend.event_processing import event_accumulator
    out = {}
    files = sorted(glob.glob(os.path.join(REPO_ROOT, run_dir, "tb_logs", "events*")))
    for f in files:
        ea = event_accumulator.EventAccumulator(f, size_guidance={"scalars": 0})
        ea.Reload()
        for tag in ea.Tags()["scalars"]:
            if tag in TB_TAGS:
                out.setdefault(tag, {}).update(
                    {int(s.step): float(s.value) for s in ea.Scalars(tag)})
    return out, len(files)


# --- literals: the same scalars, in case tensorboard is unavailable -----------
_FALLBACK = {
    "unconstrained": {
        "episode/success_rate": {1: 0.10417, 2: 0.27083, 3: 0.43750, 4: 0.54167,
                                 5: 0.43750, 6: 0.45833, 7: 0.62500, 8: 0.52083,
                                 9: 0.66667, 10: 0.60417, 11: 0.72917, 12: 0.75000,
                                 13: 0.70833, 14: 0.56250, 15: 0.68750, 16: 0.54167},
        "train/clip_loss": {1: 0.02581, 2: -0.33324, 3: -0.31573, 4: -0.33214,
                            5: -0.32975, 6: -0.32866, 7: -0.33537, 8: 0.01975,
                            9: -0.33302, 10: -0.34022, 11: -0.34441, 12: -0.32977,
                            13: -0.31971, 14: -0.34370, 15: -0.32731, 16: -0.27216},
        "ref_mse/mean": {1: 0.00423, 2: 0.00402, 3: 0.00514, 4: 0.00668, 5: 0.00822,
                         6: 0.00853, 7: 0.00829, 8: 0.00935, 9: 0.01850, 10: 0.01383,
                         11: 0.01272, 12: 0.01161, 13: 0.01126, 14: 0.01505,
                         15: 0.01895, 16: 0.04880},
        "ref_mse/log_base_ratio_mean": {1: 0.0, 2: 0.00030, 3: 0.01583, 4: 0.02714,
                                        5: 0.02902, 6: 0.03245, 7: 0.03596,
                                        8: 0.04601, 9: 0.05971, 10: 0.06144,
                                        11: 0.05832, 12: 0.06143, 13: 0.06444,
                                        14: 0.08032, 15: 0.09854, 16: 0.19282},
        "train/clipfrac": {1: 0.0, 2: 0.01468, 3: 0.04618, 4: 0.00293, 5: 0.01259,
                           6: 0.0, 7: 0.0, 8: 0.0, 9: 0.0, 10: 0.0, 11: 0.0,
                           12: 0.0, 13: 0.0, 14: 0.0, 15: 0.02489, 16: 0.12451},
    },
    "smooth0.15": {
        "episode/success_rate": {1: 0.10417, 2: 0.20833, 3: 0.25000, 4: 0.27083,
                                 5: 0.31250},
        "train/clip_loss": {1: -0.33411, 2: -0.32907, 3: -0.28361, 4: -0.31349,
                            5: -0.33990},
        "smooth/hf_mean": {1: 0.0019563, 2: 0.0039885, 3: 0.0036582, 4: 0.0038259,
                           5: 0.0036100},
        "smooth/hf_max": {1: 0.020410, 2: 0.021643, 3: 0.020460, 4: 0.039692,
                          5: 0.019300},
        "smooth/active_frac": {2: 0.20423, 3: 0.13214, 4: 0.20379, 5: 0.13830},
        "smooth/excess_mean": {2: 0.00069803, 3: 0.00046278, 4: 0.00084521,
                               5: 0.00045000},
        "smooth/loss": {2: 0.00010470, 3: 0.00006942, 4: 0.00012678,
                        5: 0.00006800},
    },
}

TB = {}
for _name, (_dir, _) in SERIES.items():
    try:
        _tb, _n = read_tb(_dir)
        if not _tb.get("episode/success_rate"):
            raise RuntimeError("no success_rate scalars found")
        TB[_name] = _tb
        print(f"{_name:14s} read {_n} event file(s), "
              f"{len(_tb['episode/success_rate'])} iterations, "
              f"{len(_tb)} of {len(TB_TAGS)} tags")
    except Exception as e:
        TB[_name] = _FALLBACK.get(_name, {})
        print(f"{_name:14s} TB read failed ({type(e).__name__}: {e}); "
              f"using embedded literals "
              f"({len(TB[_name].get('episode/success_rate', {}))} iterations)")

# hf_ref actually in force during the constrained run — read from the checkpoint
# rather than recomputed, so the dose-response below compares against the real
# frozen threshold and not a re-derivation of it.
SMOOTH_HF_REF = None
for _it in sorted(glob.glob(os.path.join(REPO_ROOT, _SMOOTH, "iter_*", "smooth_ref.json"))):
    with open(_it) as _f:
        _j = json.load(_f)
    SMOOTH_HF_REF = float(_j["hf_ref"])
    _SMOOTH_REF_META = _j
SMOOTH_HF_REF_SCALE = None
if SMOOTH_HF_REF is not None:
    SMOOTH_HF_REF_SCALE = float(_SMOOTH_REF_META.get("hf_ref_scale", 4.0))
    print(f"\nsmooth0.15 frozen hf_ref = {SMOOTH_HF_REF:.8f}  "
          f"(scale {SMOOTH_HF_REF_SCALE}, calibrated at iteration "
          f"{_SMOOTH_REF_META.get('calibrated_at_iteration')}, "
          f"source {_SMOOTH_REF_META.get('hf_ref_source')})")
    print(f"  => trainer's measured base = "
          f"{SMOOTH_HF_REF / SMOOTH_HF_REF_SCALE:.8f}")
else:
    print("\nno smooth_ref.json found (constrained run not on disk?) — the parity "
          "cell will skip the threshold comparison")

In [ ]:
# Cell 3: Load model (once)
lab = DenoisingLab(MODEL_PATH, EMBODIMENT_TAG, device=DEVICE)
print(f"Model loaded. Action horizon: {lab.action_horizon}, Action dim: {lab.action_dim}")
print(f"Inference timesteps: {lab.num_inference_timesteps}, "
      f"Timestep buckets: {lab.num_timestep_buckets}")

In [ ]:
# Cell 4: Inject LoRA ONCE into the DiT — do NOT load a checkpoint, do NOT merge.
#
# The adapter shell is injected here and reused for all 21 checkpoints: the sweep
# calls load_lora_checkpoint(...) to swap each checkpoint's A/B matrices into these
# same modules. merge_lora_weights() is deliberately skipped — merging is
# irreversible and would bake one checkpoint into the base weights, making both
# the next load and the base-model run impossible. Cost is a small per-forward
# LoRA matmul.
import sys as _sys
_grpo_dir = os.path.join(REPO_ROOT, "scripts", "grpo")
if _grpo_dir not in _sys.path:
    _sys.path.insert(0, _grpo_dir)

from lora_dit import (
    apply_lora_to_dit,
    load_lora_checkpoint,
    disabled_adapters,
    print_trainable_params,
)

apply_lora_to_dit(lab.model, rank=LORA_RANK, alpha=LORA_ALPHA, dropout=LORA_DROPOUT)

# Pin the freshly-injected LoRA Linears to the DiT's device/dtype ONCE:
# load_lora_checkpoint copies weights in place via load_state_dict, so every
# subsequent load inherits this device/dtype with no re-.to().
lab.model.action_head.model.to(device=lab.device, dtype=lab.dtype)
lab.model.eval()   # matches the trainer: train_grpo.py puts the DiT in eval mode,
                   # so dropout (0.2 in the config) is OFF and forwards are
                   # deterministic. Measuring in train mode would inject white
                   # noise straight into the quantity being measured.

print_trainable_params(lab.model)

## Dim layout and the measured rectangle

Read from the **checkpoint**, not the repo: the action key order and per-key dim
counts live in the checkpoint's normalization stats. If a checkpoint's layout ever
differs from the expected PandaOmron one, it shows up in the printout instead of
silently masking the wrong columns.

The two exclusions match `smoothness.py`'s defaults exactly, because the
constraint being sized here uses them: `gripper_close` and `control_mode` are 0/1
and thresholded at 0.5 by `PandaOmronKeyConverter.unmap_action` — a grasp *is* a
step function — and `base_motion` is gated by `control_mode` under
`HybridMobileBase`, so in arm mode it is commanded but inert.

In [ ]:
# Cell 5: Action-dim layout, read from the CHECKPOINT (self-verifying).
TAG = lab.embodiment_tag.value
_acfg = lab.modality_configs["action"]
_norm = lab.processor.state_action_processor.norm_params[TAG]["action"]

ACTION_HORIZON_VALID = len(_acfg.delta_indices)          # 16 for PandaOmron
_layout, _start = [], 0
for _key in _acfg.modality_keys:
    _d = _norm[_key]["dim"]
    _d = int(_d.item() if hasattr(_d, "item") else _d)
    _layout.append((_key, _start, _start + _d))
    _start += _d
N_VALID_DIMS = _start

DISCRETE_KEYS = {"gripper_close", "control_mode"}
INCLUDE_BASE_MOTION = False        # smoothness.DEFAULT_GATED_ACTION_KEYS

C_DIMS, C_KEYS = [], []
for _key, _lo, _hi in _layout:
    _short = _key.split(".")[-1]
    if _short in DISCRETE_KEYS:
        continue
    if _short == "base_motion" and not INCLUDE_BASE_MOTION:
        continue
    C_DIMS.extend(range(_lo, _hi))
    C_KEYS.append(_short)

# Full valid horizon, not the executed prefix: compute_fm_log_prob masks with the
# full 16x12 rectangle, so ref_mse and every advantage already cover all 16 steps,
# and 14 second-differences instead of 6 more than halves the per-row variance.
# n_action_steps is a DEPLOYMENT knob; 16 is a property of the checkpoint.
H_MEASURE = ACTION_HORIZON_VALID
N_ACTION_STEPS = 8                 # head/tail diagnostic split only
TAU_CENTERS = [0.0, 0.25, 0.35, 0.5, 0.6, 0.75]     # GRPOConfig default

print(f"embodiment={TAG}  valid_horizon={ACTION_HORIZON_VALID}  valid_dims={N_VALID_DIMS}")
print(f"raw model output = ({lab.action_horizon}, {lab.action_dim}) -> valid rectangle "
      f"[:{ACTION_HORIZON_VALID}, :{N_VALID_DIMS}]")
print("\naction dim layout (order comes from the checkpoint, NOT from the repo):")
for _key, _lo, _hi in _layout:
    _short = _key.split(".")[-1]
    _tag = "DISCRETE - excluded" if _short in DISCRETE_KEYS else (
        "excluded (INCLUDE_BASE_MOTION=False)"
        if _short == "base_motion" and not INCLUDE_BASE_MOTION else "in C")
    print(f"  [{_lo:2d}:{_hi:2d}]  {_short:28s} dim={_hi - _lo}  {_tag}")
print(f"\nC = {C_KEYS} -> {len(C_DIMS)} dims: {C_DIMS}")
print(f"h range measured: 0..{H_MEASURE - 1} (FULL valid horizon), "
      f"{max(0, H_MEASURE - 2)} second-differences")
print(f"tau centers: {TAU_CENTERS}   (tau=0 is the one the trainer constrains)")

## The measurement, and why τ=0 is the number that matters

`measure_roughness` is v4's, with three additions needed to size the constraint.

**Trainer parity at τ=0.** The production hinge evaluates the implied endpoint on
one dedicated clean forward at τ=0:

```
â(0) = ε + v_θ(ε, 0)
```

At τ=0 the FM interpolant `x_τ = (1−τ)ε + τa` **is** `ε` exactly, so
`ahat = x_tau + (1−τ)·v` computed below is bit-for-bit the same construction. This
is what makes `smooth/hf_mean` in the training logs and `HF_ahat[0]` here directly
comparable — a check the parity cell performs explicitly. (v4's `HF_ahat_mean`
averages over all six τ, which understates the τ=0 value by ~1.6× on the
finetuned checkpoints, and is *not* what the trainer constrains.)

**Pooled versus mean-of-ratios.** The trainer pools `ΣR / (6·ΣM)` over a
minibatch's rows, because per-row `HF` puts the row's own energy in the
denominator, so a near-idle chunk reports a huge ratio and dominates an unweighted
mean. Each seed here is a row, so both statistics are computable from the same
pass; reporting both quantifies how much idle-row domination the pooling is
actually removing at this batch size.

**Per-seed moments are kept.** `R` and `M` per seed per τ go into the JSON, so
every threshold and coefficient question can be re-asked later with no GPU.

In [ ]:
# Cell 6: measure_roughness — v4's measurement, plus tau=0 trainer parity,
# pooled-vs-row statistics, and per-seed moments retained for offline re-analysis.
#
# One pass over seeds computes every family, so this adds no denoise() calls:
#   * residual r = v_theta - (a - eps) at each tau  (what fm_log_prob squares)
#   * endpoint  a_hat = x_tau + (1-tau)*v           (what the hinge constrains)
#   * chunk     a itself                            (what the robot executes)
#   * path jerk of the decoded EEF path, normalized by path length
#
# Slicing to the valid rectangle happens BEFORE differencing — differencing the
# padded (50, 128) output straddles the pad boundary and returns garbage.
#
# Not batched over seeds on purpose: features has batch 1, and broadcasting a
# batch-1 vl_embeds against batch-S actions in cross-attention is a silent-
# wrongness risk.

def _d2(u):
    """Second difference along the horizon axis (axis=1) of an (N, H, D) array."""
    return u[:, 2:] - 2.0 * u[:, 1:-1] + u[:, :-2]


def _pooled_hf(R, M):
    """Trainer's statistic: sum(R) / (6*sum(M)). Equals mean(R)/(6*mean(M))."""
    R, M = np.asarray(R, float), np.asarray(M, float)
    return float(R.sum() / (6.0 * max(M.sum(), 1e-30)))


def _rowmean_hf(R, M):
    """Unweighted mean of per-row ratios — the statistic pooling replaces."""
    R, M = np.asarray(R, float), np.asarray(M, float)
    return float(np.mean(R / (6.0 * np.maximum(M, 1e-30))))


def measure_roughness(lab, features, label, n_seeds=25, tau_centers=None,
                      dims=None, h_measure=None, num_steps=4, eef_key=EEF_KEY,
                      verbose=False):
    tau_centers = TAU_CENTERS if tau_centers is None else tau_centers
    dims = C_DIMS if dims is None else dims
    H = H_MEASURE if h_measure is None else h_measure
    nb = lab.num_timestep_buckets

    per_tau = {tau: {"M": [], "R": [], "M_head": [], "R_head": [],
                     "M_tail": [], "R_tail": [], "M_ahat": [], "R_ahat": []}
               for tau in tau_centers}
    act_M, act_R, path_jerk, path_len = [], [], [], []

    he = min(N_ACTION_STEPS, H)
    head_sl, tail_sl = slice(0, max(he - 2, 0)), slice(he, max(H - 2, 0))
    spec_acc = np.zeros(H // 2 + 1)
    spec_n = 0

    for s in range(n_seeds):
        res = lab.denoise(features, seed=s, num_steps=num_steps)
        a, eps = res.action_pred, res.initial_noise
        v_target = a - eps                                    # FM velocity target

        # --- the produced chunk itself (normalized action space) ---
        a_v = a[:, :H, :].float()[:, :, dims]
        act_M.append(a_v.pow(2).mean().item())
        act_R.append(_d2(a_v).pow(2).mean().item())

        # --- scale-controlled jerk of the decoded EEF path (physical units) ---
        # Normalization is the point: LoRA paths are 28-43% longer, so RAW jerk
        # rises mechanically and cannot distinguish "rougher" from "bigger".
        d = lab.decode_raw_actions(a, features.states)
        deltas = d[eef_key][0][:, :3]                          # (T, 3)
        pos = np.concatenate([np.zeros((1, 3)), np.cumsum(deltas, axis=0)], axis=0)
        L = float(np.linalg.norm(deltas, axis=1).sum())
        J = float(np.linalg.norm(np.diff(pos, n=3, axis=0), axis=1).sum())
        path_len.append(L)
        path_jerk.append(J / max(L, 1e-9))

        for tau in tau_centers:
            x_tau = (1.0 - tau) * eps + tau * a
            # denoise_single_step computes t_cont = step_idx / num_total_steps, so
            # passing num_total_steps = num_timestep_buckets hits tau exactly.
            v, _ = lab.denoise_single_step(features, x_tau, round(tau * nb), nb)
            r = (v - v_target)[:, :H, :].float()[:, :, dims]
            d2r = _d2(r)
            # a_hat = x_tau + (1-tau)*v = a + (1-tau)*r. At tau=0, x_tau IS eps, so
            # this sits ON the sampler's path and a_hat is literally the 1-step
            # Euler endpoint — exactly fm_log_prob's smooth pass.
            ahat = (x_tau + (1.0 - tau) * v)[:, :H, :].float()[:, :, dims]
            per_tau[tau]["M_ahat"].append(ahat.pow(2).mean().item())
            per_tau[tau]["R_ahat"].append(_d2(ahat).pow(2).mean().item())
            per_tau[tau]["M"].append(r.pow(2).mean().item())
            per_tau[tau]["R"].append(d2r.pow(2).mean().item())
            _p = np.abs(np.fft.rfft(r.cpu().numpy(), axis=1)) ** 2
            spec_acc += (_p / max(_p.sum(axis=1, keepdims=True).mean(), 1e-30)
                         ).mean(axis=(0, 2))
            spec_n += 1
            if head_sl.stop > head_sl.start:
                per_tau[tau]["M_head"].append(r[:, :he].pow(2).mean().item())
                per_tau[tau]["R_head"].append(d2r[:, head_sl].pow(2).mean().item())
            if tail_sl.stop > tail_sl.start:
                per_tau[tau]["M_tail"].append(r[:, he:].pow(2).mean().item())
                per_tau[tau]["R_tail"].append(d2r[:, tail_sl].pow(2).mean().item())

    taus = list(tau_centers)
    agg = lambda k, t: (float(np.mean(per_tau[t][k])) if per_tau[t][k]
                        else float("nan"))
    M = np.array([agg("M", t) for t in taus])
    R = np.array([agg("R", t) for t in taus])
    HF = R / (6.0 * np.maximum(M, 1e-12))
    HF_head = (np.array([agg("R_head", t) for t in taus])
               / (6.0 * np.maximum([agg("M_head", t) for t in taus], 1e-12)))
    HF_tail = (np.array([agg("R_tail", t) for t in taus])
               / (6.0 * np.maximum([agg("M_tail", t) for t in taus], 1e-12)))
    HF_ahat = np.array([_pooled_hf(per_tau[t]["R_ahat"], per_tau[t]["M_ahat"])
                        for t in taus])

    # --- tau=0 block: the quantity the production hinge actually sees ----------
    i0 = taus.index(0.0) if 0.0 in taus else 0
    R0, M0 = per_tau[taus[i0]]["R_ahat"], per_tau[taus[i0]]["M_ahat"]
    rows0 = np.asarray(R0, float) / (6.0 * np.maximum(np.asarray(M0, float), 1e-30))

    out = {
        "label": label, "taus": taus, "n_seeds": n_seeds,
        "M": M.tolist(), "R": R.tolist(), "HF": HF.tolist(),
        "HF_head": HF_head.tolist(), "HF_tail": HF_tail.tolist(),
        "HF_ahat": HF_ahat.tolist(),
        "HF_ahat_mean": float(np.nanmean(HF_ahat)),
        # THE headline number: pooled endpoint HF at tau=0 == smooth/hf_mean
        "hf_tau0": float(HF_ahat[i0]),
        "hf_tau0_rowmean": _rowmean_hf(R0, M0),
        "hf_tau0_rowmin": float(rows0.min()),
        "hf_tau0_rowmax": float(rows0.max()),
        "hf_tau0_rowp90": float(np.percentile(rows0, 90)),
        "M_a": float(np.mean(act_M)),
        "M_mean": float(M.mean()), "HF_mean": float(HF.mean()),
        "HF_head_mean": float(np.nanmean(HF_head)),
        "HF_tail_mean": float(np.nanmean(HF_tail)),
        "act_HF": float(np.mean(act_R) / max(np.mean(act_M), 1e-12) / 6.0),
        "spectrum": (spec_acc / max(spec_n, 1)).tolist(),
        "path_jerk": float(np.mean(path_jerk)),
        "path_jerk_p90": float(np.percentile(path_jerk, 90)),
        "path_len": float(np.mean(path_len)),
        # per-seed moments -> every threshold question is re-askable with no GPU
        "rows": {f"{t:g}": {"R_ahat": per_tau[t]["R_ahat"],
                            "M_ahat": per_tau[t]["M_ahat"],
                            "R": per_tau[t]["R"], "M": per_tau[t]["M"]}
                 for t in taus},
    }
    if verbose:
        print(f"  M(r) mean over tau  = {out['M_mean']:.5f}  (compare ref_mse/mean)")
        print(f"  HF(r) mean over tau = {out['HF_mean']:.4f}  [0=smooth, 1=white]")
        print(f"  HF(a_hat) per tau   = " + "  ".join(
            f"{t:.2f}:{h:.4f}" for t, h in zip(taus, HF_ahat)))
        print(f"  HF(a_hat) @ tau=0   = {out['hf_tau0']:.6f}  <-- the hinge's input")
        print(f"    row-mean {out['hf_tau0_rowmean']:.6f}  "
              f"(idle-row domination = {out['hf_tau0_rowmean'] / max(out['hf_tau0'], 1e-30):.2f}x)"
              f"  row-max {out['hf_tau0_rowmax']:.6f}")
        print(f"  HF(chunk) = {out['act_HF']:.4f}   M(a) = {out['M_a']:.4f}")
        print(f"  path jerk / length  = {out['path_jerk']:.5f} "
              f"(p90 {out['path_jerk_p90']:.5f}), path length {out['path_len']:.3f}")
    return out

## The sweep

Encode the observation's backbone features **once** (the backbone is LoRA-free, and
this is the expensive part), run the base model with adapters disabled, then walk
every checkpoint, loading each adapter in place.

Cached to `RESULTS_JSON` after **every** checkpoint. Re-running this cell skips
what is already cached, so a crash at checkpoint 19 costs one checkpoint, not the
hour before it. Delete the JSON to force a full re-measure.

In [ ]:
# Cell 7: sweep driver — base + every discovered checkpoint of every series.
def discover_iters(run_dir, want=None):
    """Iteration numbers with a checkpoint on disk, ascending.

    Auto-discovered rather than hardcoded so a run that has advanced since this
    notebook was written is picked up without an edit. `want` filters.
    """
    found = []
    for p in glob.glob(os.path.join(REPO_ROOT, run_dir, "iter_*")):
        base = os.path.basename(p)
        if os.path.isfile(os.path.join(p, "lora_weights.pt")):
            try:
                found.append(int(base.split("_")[1]))
            except (IndexError, ValueError):
                pass
    found.sort()
    return [i for i in found if want is None or i in want]


def load_cache(path):
    if os.path.isfile(path):
        with open(path) as f:
            return json.load(f)
    return {}


def save_cache(cache, path):
    tmp = path + ".tmp"          # atomic-ish: a kill mid-write must not corrupt
    with open(tmp, "w") as f:
        json.dump(cache, f)
    os.replace(tmp, path)


def sweep(obs_path, series=None, n_seeds=HF_N_SEEDS, cache_path=RESULTS_JSON,
          include_base=True, verbose=False):
    """Measure base + every checkpoint of every series on ONE observation.

    Cache key is "<obs basename>|<series>|iter_XXXX" so several observations can
    share one JSON without collision.
    """
    series = SERIES if series is None else series
    cache = load_cache(cache_path)
    obs_key = os.path.basename(obs_path)

    todo = [(name, it) for name, (d, want) in series.items()
            for it in discover_iters(d, want)
            if f"{obs_key}|{name}|iter_{it:04d}" not in cache]
    n_base = 1 if (include_base and f"{obs_key}|BASE" not in cache) else 0
    fwd = (4 + len(TAU_CENTERS)) * n_seeds
    print(f"observation: {obs_key}")
    print(f"C = {C_KEYS} ({len(C_DIMS)} dims), h 0..{H_MEASURE - 1}, {n_seeds} seeds")
    print(f"to measure: {len(todo)} checkpoint(s) + {n_base} base  "
          f"({(len(todo) + n_base) * fwd} DiT forwards)")
    if not todo and not n_base:
        print("everything already cached — nothing to do.")
        return cache

    obs = DenoisingLab.load_observation(obs_path)
    features = lab.encode_features_from_sim_obs(obs)   # EXPENSIVE — once per obs
    t_all = time.time()

    if n_base:
        print(f"\n[base] adapters disabled")
        t = time.time()
        with disabled_adapters(lab.model.action_head.model):
            cache[f"{obs_key}|BASE"] = measure_roughness(
                lab, features, "BASE", n_seeds=n_seeds, verbose=True)
        save_cache(cache, cache_path)
        print(f"       {time.time() - t:.1f}s")

    for n, (name, it) in enumerate(todo, 1):
        key = f"{obs_key}|{name}|iter_{it:04d}"
        ckpt = os.path.join(REPO_ROOT, series[name][0], f"iter_{it:04d}")
        t = time.time()
        # Skip-and-continue rather than raise: one of these series may still be
        # TRAINING, so its newest lora_weights.pt can be half-written when we get
        # to it. Dying there would strand the remaining checkpoints for no reason
        # (every finished one is already cached).
        try:
            load_lora_checkpoint(lab.model, ckpt)
            r = measure_roughness(lab, features, key, n_seeds=n_seeds,
                                  verbose=verbose)
        except Exception as e:
            print(f"[{n:2d}/{len(todo)}] {name:14s} iter_{it:04d}  SKIPPED "
                  f"({type(e).__name__}: {e})")
            continue
        cache[key] = r
        save_cache(cache, cache_path)
        el = time.time() - t
        eta = (time.time() - t_all) / n * (len(todo) - n)
        print(f"[{n:2d}/{len(todo)}] {name:14s} iter_{it:04d}  "
              f"hf(tau=0)={r['hf_tau0']:.6f}  chunkHF={r['act_HF']:.4f}  "
              f"pathjerk={r['path_jerk']:.4f}  {el:.1f}s  ETA {eta / 60:.1f}min")

    print(f"\ntotal {(time.time() - t_all) / 60:.1f} min -> {cache_path}")
    return cache


RESULTS = sweep(OBS_MAIN)

## Trainer parity

The constrained run logged `smooth/hf_mean` live — the pooled τ=0 endpoint HF over
its own minibatch rows. This notebook computes the same statistic from 25 seeds on
one observation. They are not identical measurements (the trainer pools rows drawn
from many observations along the rollout; this is one observation), so they should
*agree in scale*, not to the digit.

Two things this catches. A large disagreement means the wiring measures something
other than what was intended — the failure mode the whole design is exposed to,
since a silently-contaminated HF would look like a working constraint. Agreement
means the offline sweep below can be trusted to size a threshold that the trainer
will then reproduce.

In [ ]:
# Cell 8: parity — notebook tau=0 pooled HF vs the run's own smooth/hf_mean.
def series_curve(results, series_name, field, obs_key=None):
    """{iteration: value} for one field of one series, ascending."""
    obs_key = os.path.basename(OBS_MAIN) if obs_key is None else obs_key
    out = {}
    for k, v in results.items():
        parts = k.split("|")
        if len(parts) == 3 and parts[0] == obs_key and parts[1] == series_name:
            out[int(parts[2].split("_")[1])] = v[field] if field in v else None
    return dict(sorted(out.items()))


BASE = RESULTS.get(f"{os.path.basename(OBS_MAIN)}|BASE")
base_hf0 = BASE["hf_tau0"] if BASE else float("nan")

tb_hf = TB.get("smooth0.15", {}).get("smooth/hf_mean", {})
nb_hf = series_curve(RESULTS, "smooth0.15", "hf_tau0")

print(f"base (no LoRA) pooled HF(a_hat) @ tau=0 = {base_hf0:.6f}")
if SMOOTH_HF_REF:
    _tb_base = SMOOTH_HF_REF / SMOOTH_HF_REF_SCALE
    print(f"trainer's calibrated base           = {_tb_base:.6f}  "
          f"(hf_ref {SMOOTH_HF_REF:.6f} / scale {SMOOTH_HF_REF_SCALE})")
    print(f"  agreement: {base_hf0 / max(_tb_base, 1e-30):.3f}x   "
          f"(within ~1.2x = wiring confirmed)")

if tb_hf and nb_hf:
    print(f"\n{'iter':>5} {'notebook hf(tau=0)':>19} {'TB smooth/hf_mean':>19} {'ratio':>7}")
    for it in sorted(set(tb_hf) & set(nb_hf)):
        print(f"{it:>5} {nb_hf[it]:>19.6f} {tb_hf[it]:>19.6f} "
              f"{nb_hf[it] / max(tb_hf[it], 1e-30):>7.2f}")
    print("\nThe trainer pools rows from many observations along the rollout; this")
    print("notebook uses 25 seeds on ONE observation. Same scale = the hinge is")
    print("reading the field it was designed to read.")
else:
    print("\n(no overlapping iterations to compare — run the sweep first)")

## Does roughness lead the collapse?

This is the question that decides whether a hinge can *prevent* anything. A
constraint that fires at the same iteration success falls is a detector, not a
guardrail — the only useful version is one whose instrument moves first.

For each candidate instrument the table reports:

* **base / peak / max** — value at base weights, at the iteration of peak success,
  and the trajectory maximum.
* **ρ_all** — Spearman correlation with success rate over the whole trajectory.
  Expect this to be **positive and useless**: success climbs from 0.10 to 0.75 over
  the first twelve iterations while roughness climbs monotonically, so both rise
  together and the correlation measures "training progressed", not "roughness
  hurt". Reported only so the confound is visible rather than hidden.
* **ρ_post** — the same correlation restricted to the post-peak window
  (`iter >= peak - 2`). Strongly negative means the instrument keeps rising while
  success falls.
* **ρ→ΔSR** — the predictive one, and the reason this table exists: the
  instrument's value at iteration `i` against the success *change* from `i` to
  `i+1`. Strongly negative means roughness now forecasts a success drop next
  iteration, which is the only correlation a preventive constraint can act on.
* **onset@k×** — first iteration the instrument exceeds `k ×` its base value.
  Compared against the peak-success iteration, this is the **lead time** available
  to a threshold set at `k × base`.

In [ ]:
# Cell 9: instrument comparison — correlation with success and lead time.
def spearman(x, y):
    """Rank correlation, no scipy. Ties get average ranks."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    if len(x) < 3:
        return float("nan")

    def rank(v):
        order = np.argsort(v, kind="mergesort")
        r = np.empty(len(v), float)
        r[order] = np.arange(len(v), dtype=float)
        # average ranks within tie groups
        i = 0
        while i < len(v):
            j = i
            while j + 1 < len(v) and v[order[j + 1]] == v[order[i]]:
                j += 1
            if j > i:
                r[order[i:j + 1]] = np.mean(r[order[i:j + 1]])
            i = j + 1
        return r

    rx, ry = rank(x), rank(y)
    rx, ry = rx - rx.mean(), ry - ry.mean()
    den = math.sqrt(float((rx ** 2).sum() * (ry ** 2).sum()))
    return float((rx * ry).sum() / den) if den > 0 else float("nan")


INSTRUMENTS = [
    ("hf_tau0",      "endpoint HF @ tau=0   (the hinge's input)"),
    ("HF_ahat_mean", "endpoint HF mean over tau"),
    ("act_HF",       "chunk HF (4-step generated a)"),
    ("path_jerk",    "EEF path jerk / path length"),
    ("M_mean",       "M(r) = residual energy  (~ref_mse)"),
    ("HF_mean",      "residual HF (spectrum only)"),
]
ONSET_K = [2, 4, 10, 25]
SERIES_FOR_SIZING = "unconstrained"


def rho_variants(curve, sr, peak_it):
    """(rho_all, rho_post_peak, rho_vs_next_delta_SR) for one instrument.

    rho_all is confounded: over a run that is still learning, success and
    roughness both rise, so a positive value says nothing about harm. The other
    two isolate the decline — rho_post over the post-peak window, and rho_dsr
    against the NEXT iteration's success change, which is the only form a
    preventive constraint can act on.
    """
    its = sorted(set(curve) & set(sr))
    rho_all = spearman([curve[i] for i in its], [sr[i] for i in its])
    post = [i for i in its if peak_it is not None and i >= peak_it - 2]
    rho_post = (spearman([curve[i] for i in post], [sr[i] for i in post])
                if len(post) >= 3 else float("nan"))
    pairs = [(curve[i], sr[i + 1] - sr[i]) for i in sorted(curve)
             if i in sr and (i + 1) in sr]
    rho_dsr = (spearman([a for a, _ in pairs], [b for _, b in pairs])
               if len(pairs) >= 3 else float("nan"))
    return rho_all, rho_post, rho_dsr

sr = TB.get(SERIES_FOR_SIZING, {}).get("episode/success_rate", {})
peak_it = max(sr, key=sr.get) if sr else None
print(f"sizing series: {SERIES_FOR_SIZING}")
if peak_it:
    print(f"peak success = {sr[peak_it]:.3f} at iteration {peak_it}; "
          f"post-peak: " + " ".join(f"{i}:{sr[i]:.2f}" for i in sorted(sr) if i > peak_it))
print()

hdr = (f"{'instrument':40s} {'base':>9} {'@peak':>9} {'max':>9} {'max/base':>8} "
       f"{'rho_all':>8} {'rho_post':>8} {'rho>dSR':>8} "
       + " ".join(f"{'on@' + str(k) + 'x':>7}" for k in ONSET_K))
print(hdr); print("-" * len(hdr))
CURVES, RHOS = {}, {}
for field, desc in INSTRUMENTS:
    c = {k: v for k, v in series_curve(RESULTS, SERIES_FOR_SIZING, field).items()
         if v is not None and np.isfinite(v)}
    CURVES[field] = c
    if not c or BASE is None:
        print(f"{desc:40s}   (no data)")
        continue
    b = BASE[field]
    its = sorted(c)
    r_all, r_post, r_dsr = rho_variants(c, sr, peak_it)
    RHOS[field] = (r_all, r_post, r_dsr)
    onsets = []
    for k in ONSET_K:
        hit = next((i for i in its if c[i] > k * b), None)
        onsets.append("never" if hit is None else str(hit))
    mx = max(c.values())
    print(f"{desc:40s} {b:9.5f} "
          f"{(c.get(peak_it, float('nan'))):9.5f} {mx:9.5f} {mx / max(b, 1e-30):8.1f} "
          f"{r_all:+8.2f} {r_post:+8.2f} {r_dsr:+8.2f} "
          + " ".join(f"{o:>7}" for o in onsets))

print("\nLEAD TIME = (peak-success iteration) - (onset iteration). Positive means a")
print("threshold at k x base fires BEFORE success turns over, which is the only")
print("configuration in which the hinge can prevent rather than annotate.")

## Dose–response: what `(smooth_coef, smooth_hf_ref_scale)` actually does

The hinge is

```
L = smooth_coef * relu( HF_pooled(tau=0) - smooth_hf_ref )
    smooth_hf_ref = smooth_hf_ref_scale * base_HF
```

so it is fully determined by the measured `HF(iteration)` curve plus the two knobs.
Because it is a hinge, "make the term matter now" and "make the term matter at
collapse" are far apart in coefficient — at `smooth_coef=0.15` the run logged the
term at 0.03% of `|clip_loss|` while the field sat at 0.5× threshold. That is not
evidence the coefficient is small; it is evidence the constraint had nothing to do.
What follows separates the two.

`|clip_loss|` reference is the **median over iterations where it is negative**. The
first iteration of each training process reports a positive `clip_loss` (+0.026 at
iter 1, +0.020 at iter 8) because the PAWS target ratio `k` is not yet at its
target there; including those in a denominator would inflate the reported share by
an order of magnitude at exactly two arbitrary iterations.

Read the grid for two properties at once:
* **quiet while improving** — the term should be ≈0 while success is still rising,
  or it is taxing the policy for nothing.
* **loud before the turn** — it needs real authority at or before the peak-success
  iteration, not 17× later.

In [ ]:
# Cell 10: dose-response over (smooth_coef, smooth_hf_ref_scale).
COEF_GRID = [0.15, 0.3, 0.5, 1.0, 2.0, 4.0]
SCALE_GRID = [1.5, 2.0, 4.0, 8.0]
QUIET_SHARE = 0.005      # <=0.5% of |clip_loss| counts as "silent"
LOUD_SHARE = 0.05        # >=5% counts as "real authority"

cl = TB.get(SERIES_FOR_SIZING, {}).get("train/clip_loss", {})
_neg = [abs(v) for v in cl.values() if v < 0]
CLIP_REF = float(np.median(_neg)) if _neg else 0.33
hf = CURVES["hf_tau0"]
its = sorted(hf)
# "still improving" = every iteration at least 3 before the peak; a term that is
# loud here is buying smoothness with capability the run had not yet banked.
improving = [i for i in its if peak_it is None or i <= peak_it - 3]

print(f"base HF(tau=0) = {base_hf0:.6f}   |clip_loss| ref = {CLIP_REF:.4f} "
      f"(median of {len(_neg)} negative iters)")
print(f"peak success at iter {peak_it}; 'still improving' = iters "
      f"{improving[0] if improving else '-'}..{improving[-1] if improving else '-'}\n")

hdr = (f"{'coef':>6} {'scale':>6} {'hf_ref':>9} {'onset':>6} {'lead':>5} "
       f"{'share@peak':>11} {'max share':>10} {'quiet?':>7} {'loud by peak?':>14}")
print(hdr); print("-" * len(hdr))
GRID = {}
for scale in SCALE_GRID:
    thr = scale * base_hf0
    for coef in COEF_GRID:
        share = {i: coef * max(0.0, hf[i] - thr) / CLIP_REF for i in its}
        onset = next((i for i in its if share[i] >= QUIET_SHARE), None)
        quiet = all(share[i] < QUIET_SHARE for i in improving)
        loud = (peak_it is not None
                and any(share[i] >= LOUD_SHARE for i in its if i <= peak_it))
        GRID[(coef, scale)] = share
        print(f"{coef:6.2f} {scale:6.1f} {thr:9.5f} "
              f"{('never' if onset is None else onset):>6} "
              f"{('-' if (onset is None or peak_it is None) else peak_it - onset):>5} "
              f"{share.get(peak_it, float('nan')):11.1%} {max(share.values()):10.1%} "
              f"{('yes' if quiet else 'NO'):>7} {('yes' if loud else 'no'):>14}")

print(f"\nquiet?  = term < {QUIET_SHARE:.1%} of |clip_loss| at every iteration while")
print(f"          success was still improving (iters <= peak-3)")
print(f"loud by peak? = term reaches {LOUD_SHARE:.0%} at or before the peak iteration")
print("lead    = peak iteration - onset iteration (iterations of warning)")
print("\nThe setting you want is the cheapest coefficient that is BOTH quiet and loud")
print("by peak. If no row achieves both, the hinge cannot be tuned into a preventer")
print("on this trajectory and the threshold scale is the binding constraint, not the")
print("coefficient — see the recommendation cell.")

In [ ]:
# Cell 11: figures.
_S = [s for s in SERIES if series_curve(RESULTS, s, "hf_tau0")]
fig, axes = plt.subplots(2, 3, figsize=(21, 10), constrained_layout=True)

# (0,0) the roughening curve, both series, with candidate thresholds
ax = axes[0, 0]
for s in _S:
    c = series_curve(RESULTS, s, "hf_tau0")
    ax.plot(sorted(c), [c[i] for i in sorted(c)], "-o", ms=4, label=s)
ax.axhline(base_hf0, color="k", ls="-", lw=1, label=f"base {base_hf0:.4f}")
for k, ls in zip([2, 4, 10], [":", "--", "-."]):
    ax.axhline(k * base_hf0, color="gray", ls=ls, lw=0.9, label=f"{k}x base")
if SMOOTH_HF_REF:
    ax.axhline(SMOOTH_HF_REF, color="crimson", ls="--", lw=1.2,
               label=f"hf_ref in force {SMOOTH_HF_REF:.4f}")
if peak_it:
    ax.axvline(peak_it, color="green", ls=":", lw=1, label=f"peak SR (it {peak_it})")
ax.set_yscale("log"); ax.set_xlabel("iteration"); ax.set_ylabel("pooled HF(a_hat) @ tau=0")
ax.set_title("endpoint roughness — the hinge's input", fontsize=10)

# (0,1) roughness vs success on one axis
ax = axes[0, 1]
for s in _S:
    c = series_curve(RESULTS, s, "hf_tau0")
    ax.plot(sorted(c), [c[i] for i in sorted(c)], "-o", ms=4, label=f"HF {s}")
ax.set_yscale("log"); ax.set_xlabel("iteration"); ax.set_ylabel("HF (log)")
ax2 = ax.twinx()
for s in _S:
    q = TB.get(s, {}).get("episode/success_rate", {})
    if q:
        ax2.plot(sorted(q), [q[i] for i in sorted(q)], "--s", ms=3, alpha=0.65,
                 label=f"SR {s}")
ax2.set_ylabel("success rate")
ax.set_title("does roughness lead the decline?", fontsize=10)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=7, loc="upper left")

# (0,2) the other instruments, each normalized to its own base value
ax = axes[0, 2]
for field, desc in INSTRUMENTS:
    c = CURVES.get(field) or {}
    if not c or BASE is None or not np.isfinite(BASE[field]) or BASE[field] == 0:
        continue
    ax.plot(sorted(c), [c[i] / BASE[field] for i in sorted(c)], "-o", ms=3,
            label=desc.split("(")[0].strip())
if peak_it:
    ax.axvline(peak_it, color="green", ls=":", lw=1)
ax.axhline(1.0, color="k", lw=0.8)
ax.set_yscale("log"); ax.set_xlabel("iteration"); ax.set_ylabel("value / base value")
ax.set_title(f"every instrument, normalized to base — {SERIES_FOR_SIZING}", fontsize=10)
ax.legend(fontsize=7)

# (1,0) energy: M(r) against the run's own ref_mse (they measure the same thing)
ax = axes[1, 0]
c = CURVES.get("M_mean") or {}
if c:
    ax.plot(sorted(c), [c[i] for i in sorted(c)], "-o", ms=4, label="M(r) measured here")
q = TB.get(SERIES_FOR_SIZING, {}).get("ref_mse/mean", {})
if q:
    ax.plot(sorted(q), [q[i] for i in sorted(q)], "--s", ms=3, label="ref_mse/mean (TB)")
ax.set_yscale("log"); ax.set_xlabel("iteration"); ax.set_ylabel("M")
ax.set_title("energy cross-check: M(r) vs ref_mse/mean", fontsize=10)
ax.legend(fontsize=7)

# (1,1) dose-response heat map: share of |clip_loss| at the peak iteration
ax = axes[1, 1]
grid = np.array([[GRID[(cf, sc)].get(peak_it, np.nan) for cf in COEF_GRID]
                 for sc in SCALE_GRID])
im = ax.imshow(grid * 100, aspect="auto", origin="lower", cmap="viridis")
ax.set_xticks(range(len(COEF_GRID))); ax.set_xticklabels(COEF_GRID)
ax.set_yticks(range(len(SCALE_GRID))); ax.set_yticklabels(SCALE_GRID)
ax.set_xlabel("smooth_coef"); ax.set_ylabel("smooth_hf_ref_scale")
for i in range(len(SCALE_GRID)):
    for j in range(len(COEF_GRID)):
        ax.text(j, i, f"{grid[i, j] * 100:.1f}", ha="center", va="center",
                color="w", fontsize=8)
ax.set_title(f"term as % of |clip_loss| at peak-SR iter {peak_it}", fontsize=10)
plt.colorbar(im, ax=ax, label="%")

# (1,2) pooled vs mean-of-ratios — how much idle-row domination pooling removes
ax = axes[1, 2]
for s in _S:
    p = series_curve(RESULTS, s, "hf_tau0")
    m = series_curve(RESULTS, s, "hf_tau0_rowmean")
    x = sorted(set(p) & set(m))
    if x:
        ax.plot(x, [m[i] / max(p[i], 1e-30) for i in x], "-o", ms=4, label=s)
ax.axhline(1.0, color="k", lw=0.8)
ax.set_xlabel("iteration"); ax.set_ylabel("row-mean HF / pooled HF")
ax.set_title("idle-row domination removed by pooling (>1 = pooling matters)",
             fontsize=10)
ax.legend(fontsize=7)

for a in axes.ravel():
    a.grid(True, alpha=0.3)
plt.show()

## Is a single global `hf_ref` safe?

v4's `report()` printed a warning that a global scalar threshold is not safe,
because the base profile varies several-fold across observations. That is checkable
directly: measure the **base** model's τ=0 endpoint HF on each observation and look
at the spread.

It matters differently for the two run types. A run pinned to one init state via
`--init-state-npz-path` only ever sees observations near that state, so one scalar
is fine — and the parity cell above confirms the trainer's calibration landed on
step010's base value. A `full_run` spanning many init states pools across the whole
spread, so a scalar calibrated at 4× the pooled mean can sit *below* a
smoother-than-average observation's base value, making the term read as active on
an unroughened field.

In [ ]:
# Cell 12: cross-observation base spread + a checkpoint spot check.
# Only base + a few checkpoints per extra observation: the point is the spread of
# the THRESHOLD across observations, which base alone determines, plus enough
# checkpoints to confirm the roughening ratio is not an artifact of step010.
SPOT_ITERS = None       # None -> pick first / middle / last automatically

_ctl_iters = discover_iters(SERIES[SERIES_FOR_SIZING][0])
if SPOT_ITERS is None and _ctl_iters:
    SPOT_ITERS = sorted({_ctl_iters[0], _ctl_iters[len(_ctl_iters) // 2],
                         _ctl_iters[-1]})
print(f"spot-check iterations: {SPOT_ITERS}\n")

for _obs in OBS_EXTRA:
    if not os.path.isfile(_obs):
        print(f"SKIP (missing): {_obs}")
        continue
    RESULTS = sweep(_obs, series={SERIES_FOR_SIZING: (SERIES[SERIES_FOR_SIZING][0],
                                                      SPOT_ITERS)})
    print()

print("=" * 78)
print("BASE endpoint HF @ tau=0 by observation — this IS the threshold's base")
print("=" * 78)
_bases = {}
for _obs in [OBS_MAIN] + OBS_EXTRA:
    k = f"{os.path.basename(_obs)}|BASE"
    if k in RESULTS:
        _bases[os.path.basename(_obs)] = RESULTS[k]["hf_tau0"]
for k, v in _bases.items():
    print(f"  {k:34s} {v:.6f}")
if len(_bases) > 1:
    lo, hi = min(_bases.values()), max(_bases.values())
    print(f"\n  spread = {hi / max(lo, 1e-30):.2f}x  (lo {lo:.6f}, hi {hi:.6f})")
    print(f"  a scalar hf_ref calibrated on the LOW observation at scale 4.0 "
          f"= {4 * lo:.6f}")
    print(f"  ... which is {4 * lo / max(hi, 1e-30):.2f}x the HIGH observation's "
          f"BASE value.")
    if 4 * lo < hi:
        print("  -> BELOW it: on that observation the term would read as active on an")
        print("     unroughened field. Per-observation or per-tau refs needed for a")
        print("     multi-init-state run; fine for a single pinned init state.")
    else:
        print("  -> above it: one scalar at scale 4.0 clears every observation's base.")

print("\nroughening ratio by observation (max measured iter / base):")
for _obs in [OBS_MAIN] + OBS_EXTRA:
    ok = os.path.basename(_obs)
    c = series_curve(RESULTS, SERIES_FOR_SIZING, "hf_tau0", obs_key=ok)
    b = RESULTS.get(f"{ok}|BASE", {}).get("hf_tau0")
    if c and b:
        print(f"  {ok:34s} {max(c.values()) / b:7.1f}x   "
              f"(base {b:.6f} -> max {max(c.values()):.5f})")

## Recommendation

In [ ]:
# Cell 13: pull it together into a concrete recommendation.
print("=" * 78)
print("SMOOTHING DESIGN — READ-OUT")
print("=" * 78)

hf = CURVES["hf_tau0"]
its = sorted(hf)
print(f"\n1. ROUGHENING CURVE ({SERIES_FOR_SIZING}, {len(its)} iterations)")
print(f"   base {base_hf0:.6f}  ->  max {max(hf.values()):.5f} "
      f"({max(hf.values()) / max(base_hf0, 1e-30):.0f}x)")
print("   " + "  ".join(f"{i}:{hf[i]:.4f}" for i in its))

print(f"\n2. INSTRUMENT  (judged on rho>dSR: does it forecast the NEXT drop?)")
_r = RHOS.get("hf_tau0", (float("nan"),) * 3)
print(f"   endpoint HF @ tau=0 (in use):  rho_all {_r[0]:+.2f}  "
      f"rho_post {_r[1]:+.2f}  rho>dSR {_r[2]:+.2f}")
_ranked = sorted(((f, v) for f, v in RHOS.items() if np.isfinite(v[2])),
                 key=lambda t: t[1][2])
for f, v in _ranked[:3]:
    print(f"     {f:16s} rho>dSR {v[2]:+.2f}  rho_post {v[1]:+.2f}"
          + ("   <-- in use" if f == "hf_tau0" else ""))
if _ranked and _ranked[0][0] != "hf_tau0":
    print(f"   -> {_ranked[0][0]} forecasts the decline better than the quantity the")
    print(f"      hinge constrains. That outranks any coefficient choice: switching")
    print(f"      instrument is a bigger lever than retuning smooth_coef.")
elif _ranked:
    print("   -> the hinge is already on the best-forecasting instrument measured.")
if _ranked and _ranked[0][1][2] > -0.2:
    print("   !! NO instrument forecasts the next-iteration drop (best rho>dSR is")
    print("      near zero). On this trajectory the collapse is not predictable from")
    print("      intra-chunk roughness, so no hinge setting can PREVENT it — treat")
    print("      the constraint as damage control and check the inter-chunk seam")
    print("      instead (chunk k step 7 -> chunk k+1 step 0).")

viable = [(cf, sc) for (cf, sc), sh in GRID.items()
          if all(sh[i] < QUIET_SHARE for i in improving)
          and peak_it is not None
          and any(sh[i] >= LOUD_SHARE for i in its if i <= peak_it)]
print(f"\n3. (coef, scale) SETTINGS THAT ARE BOTH QUIET-WHILE-IMPROVING AND "
      f"LOUD-BY-PEAK")
if viable:
    viable.sort(key=lambda p: (p[0], p[1]))
    for cf, sc in viable:
        sh = GRID[(cf, sc)]
        onset = next((i for i in its if sh[i] >= QUIET_SHARE), None)
        lead = "-" if onset is None else str(peak_it - onset)
        print(f"   smooth_coef={cf:<5} smooth_hf_ref_scale={sc:<5} "
              f"onset it {onset}, lead {lead}, "
              f"peak share {sh.get(peak_it, float('nan')):.1%}, "
              f"max {max(sh.values()):.1%}")
    cf, sc = viable[0]
    print(f"\n   -> cheapest viable: --smooth-coef {cf} "
          f"--smooth-hf-ref-scale {sc}")
else:
    print("   NONE. Every setting is either loud while success is still improving or")
    print("   silent until after the peak. Options, in order of preference:")
    print("     a) lower smooth_hf_ref_scale further (the threshold, not the coef, is")
    print("        binding) and re-run this grid;")
    print("     b) switch instrument if section 2 found a better one;")
    print("     c) accept a reactive constraint: size the coef for authority AT the")
    print("        collapse and treat it as damage control, not prevention.")

print(f"\n4. WHAT smooth_coef=0.15 / scale=4.0 ACTUALLY DID")
sh = GRID.get((0.15, 4.0))
if sh:
    onset = next((i for i in its if sh[i] >= QUIET_SHARE), None)
    print(f"   threshold {4.0 * base_hf0:.6f}, onset "
          f"{'never' if onset is None else f'iter {onset}'}, "
          f"peak share {sh.get(peak_it, float('nan')):.2%}, "
          f"max share {max(sh.values()):.1%}")

c_new = series_curve(RESULTS, "smooth0.15", "hf_tau0")
c_old = CURVES["hf_tau0"]
both = sorted(set(c_new) & set(c_old))
print(f"\n5. DID THE TERM BEND THE CURVE? (iterations measured in both runs)")
if both:
    print(f"   {'iter':>5} {'unconstrained':>14} {'smooth0.15':>11} {'ratio':>7}")
    for i in both:
        print(f"   {i:>5} {c_old[i]:>14.6f} {c_new[i]:>11.6f} "
              f"{c_new[i] / max(c_old[i], 1e-30):>7.2f}")
    print("   ratio < 1 means the constrained run's field is smoother at the same")
    print("   iteration. With the term at ~0.03% of the objective, expect ~1.0 — and")
    print("   a ratio near 1.0 means the success-rate difference between the two runs")
    print("   is NOT attributable to the smoothness term.")
else:
    print("   no overlapping iterations measured yet.")
print("\n" + "=" * 78)